# 1. Memuat Dataset

Tahap pertama dalam proses analisis adalah memuat dataset menggunakan library **Pandas**. Dataset `data.zip` dibaca menggunakan fungsi `read_csv()` dengan pemisah data berupa titik koma (`;`).

Setelah dataset berhasil dimuat, dilakukan beberapa pemeriksaan awal, yaitu:

- Menampilkan 5 baris pertama menggunakan `head()` untuk melihat contoh isi data.
- Menampilkan ukuran dataset menggunakan `shape` untuk mengetahui jumlah baris dan kolom.
- Menampilkan informasi struktur dataset menggunakan `info()` untuk mengetahui nama kolom, jumlah data yang terisi, serta tipe data pada setiap kolom.

Pemeriksaan awal ini dilakukan untuk memahami kondisi dataset sebelum masuk ke tahap *data cleaning* dan analisis lebih lanjut.

In [21]:
import pandas as pd

# Membaca dataset
df = pd.read_csv("data.zip", sep=";")

# Melihat 5 baris pertama
display(df.head())

# Melihat jumlah baris dan kolom
print("Ukuran dataset:", df.shape)

# Melihat struktur dan tipe data
df.info()

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,order_date
0,ORD_0000001,2.0,2000,0,0,Celengan,1,Selesai,Tidak dibatalkan,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,2024-04-01
1,ORD_0000002,1.0,500,0,0,Celengan,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,2024-04-01
2,ORD_0000003,1.0,500,0,0,Celengan,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,2024-04-01
3,ORD_0000004,2.0,400,0,0,Mangkok Sambal / Saus,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,2024-04-01
4,ORD_0000005,3.0,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,2024-04-01


Ukuran dataset: (18868, 19)
<class 'pandas.DataFrame'>
RangeIndex: 18868 entries, 0 to 18867
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   order_id                            18868 non-null  str    
 1   total_qty                           18585 non-null  float64
 2   total_weight_gr                     18868 non-null  int64  
 3   total_returned_qty                  18868 non-null  int64  
 4   Total Diskon                        18868 non-null  int64  
 5   product_categories                  18868 non-null  str    
 6   num_product_categories              18868 non-null  int64  
 7   Status Pesanan                      18868 non-null  str    
 8   Alasan Pembatalan                   18868 non-null  str    
 9   Opsi Pengiriman                     18868 non-null  str    
 10  Metode Pembayaran                   18585 non-null  str    
 11  Kota/Kabupaten          

# 2. Menentukan Kolom Relevan untuk Analisis

Sebelum lanjut ngolah data, kita petain dulu pertanyaan bisnis sama kolom-kolom yang mau dipakai:

* **Pendapatan Tertinggi per Produk**
  * **Kolom:** `product_categories`, `total_pembayaran`, `status_pesanan`
  * **Tujuan:** Buat ngeliat kategori atau produk mana yang nyumbang omset paling gede.

* **Tren Penjualan Tahunan**
  * **Kolom:** `order_date`, `total_pembayaran`, `status_pesanan`
  * **Tujuan:** Ngerekap total nilai transaksi dari tahun ke tahun biar kelihatan grafiknya naik atau turun.

* **Kategori Terlaris di Kabupaten Bogor**
  * **Kolom:** `product_categories`, `total_qty`, `kota_kabupaten`, `status_pesanan`
  * **Tujuan:** Ngerinci barang apa aja yang paling laku khusus buat daerah Kabupaten Bogor.

**Catatan Poin Penting:**
* Isi kolom `product_categories` itu nama kategori (bukan selalu 1 item produk spesifik). Kalau ada 1 transaksi yang isinya beberapa kategori gabungan, entar perlu kita *split* dulu datanya pas analisis.
* Pastiin data di `total_pembayaran` udah bersih dan beneran ngegambarin nominal yang masuk.

# 3. Pemeriksaan Kondisi dan Karakteristik Dataset

Setelah dataset berhasil dimuat, dilakukan pemeriksaan lebih lanjut untuk memahami kondisi dan karakteristik data. Pemeriksaan ini meliputi **missing values, data duplikat, status pesanan, kategori produk, wilayah, rentang tanggal, serta statistik deskriptif pada data numerik**.

Pemeriksaan *missing values* dilakukan untuk mengetahui jumlah data yang kosong pada setiap kolom. Informasi ini digunakan sebagai dasar untuk menentukan metode penanganan data yang kosong pada tahap *data cleaning*.

Selanjutnya, jumlah data duplikat diperiksa untuk memastikan tidak terdapat transaksi yang tercatat lebih dari satu kali.

Kolom `Status Pesanan` diperiksa untuk mengetahui berbagai status transaksi yang terdapat dalam dataset. Informasi ini penting karena analisis pendapatan dan penjualan perlu mempertimbangkan status transaksi, terutama untuk membedakan pesanan yang berhasil dan pesanan yang dibatalkan.

Distribusi `product_categories` juga diperiksa untuk mengetahui kategori produk yang paling sering muncul dalam dataset. Pemeriksaan ini berkaitan dengan analisis mengenai produk atau kategori yang memiliki penjualan dan pendapatan tertinggi.

Kolom `Kota/Kabupaten` diperiksa untuk mengetahui wilayah yang terdapat dalam dataset. Hal ini diperlukan karena salah satu analisis akan berfokus pada transaksi yang berasal dari **Kabupaten Bogor**.

Rentang tanggal pesanan juga diperiksa untuk mengetahui periode transaksi yang tersedia. Informasi ini menjadi dasar dalam melakukan analisis tren penjualan berdasarkan tahun.

Terakhir, statistik deskriptif pada kolom `total_qty`, `Total Pembayaran`, dan `Total Diskon` digunakan untuk melihat gambaran umum data numerik, seperti jumlah data, nilai rata-rata, nilai minimum, maksimum, serta persebaran data.

Hasil pemeriksaan ini kemudian digunakan sebagai dasar untuk menentukan langkah *data cleaning* yang sesuai dengan kebutuhan analisis.

In [22]:
# Memeriksa missing values
print("Missing values:")
display(df.isnull().sum().to_frame("Jumlah Missing"))

# Memeriksa jumlah duplikat
print("Jumlah baris duplikat:", df.duplicated().sum())

# Memeriksa status pesanan
display(df["Status Pesanan"].value_counts(dropna=False))

# Memeriksa kategori yang paling sering muncul
display(df["product_categories"].value_counts().head(10))

# Memeriksa wilayah yang tersedia
display(
    df["Kota/Kabupaten"]
    .value_counts()
    .head(10)
)

# Memeriksa rentang tanggal
print("Tanggal awal:", df["order_date"].min())
print("Tanggal akhir:", df["order_date"].max())

# Statistik numerik
display(
    df[[
        "total_qty",
        "Total Pembayaran",
        "Total Diskon"
    ]].describe()
)

Missing values:


,Jumlah Missing
order_id,0
total_qty,283
total_weight_gr,0
total_returned_qty,0
Total Diskon,0
product_categories,0
num_product_categories,0
Status Pesanan,0
Alasan Pembatalan,0
Opsi Pengiriman,0


Jumlah baris duplikat: 0


Status Pesanan
Selesai                                                                                   16045
Batal                                                                                      2573
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun P

product_categories
Celengan                  5083
Mangkok Sambal / Saus     3253
Aksesoris Pintu           2365
Nampan / Tray             1140
Seal / Baut / Roof         586
Rak / Rak Serbaguna        573
Baskom / Mangkok Besar     536
Lunch Box / Rantang        492
Other                      473
Keranjang                  457
Name: count, dtype: int64

Kota/Kabupaten
KAB. BOGOR                1082
KAB. TANGERANG            1027
KOTA TANGERANG             915
KAB. BEKASI                750
KOTA TANGERANG SELATAN     740
KOTA JAKARTA BARAT         681
KOTA JAKARTA SELATAN       634
KOTA JAKARTA TIMUR         616
KOTA BEKASI                564
KOTA DEPOK                 541
Name: count, dtype: int64

Tanggal awal: 01/02/2025
Tanggal akhir: 31/08/2025


,total_qty,Total Pembayaran,Total Diskon
count,18585.000000,1.886800e+04,18868.000000
mean,2.565940,5.099066e+04,424.220691
std,7.799706,1.426189e+05,10110.492750
min,1.000000,0.000000e+00,0.000000
25%,1.000000,1.367500e+04,0.000000
50%,1.000000,2.180000e+04,0.000000
75%,2.000000,4.231125e+04,0.000000
max,256.000000,3.076000e+06,700000.000000


# 4. Menghapus Data Duplikat dan Memeriksa Missing Values

Tahap pertama dalam *data cleaning* adalah menghapus data yang terduplikasi. Data duplikat dapat menyebabkan satu transaksi dihitung lebih dari satu kali sehingga dapat memengaruhi hasil perhitungan pendapatan maupun jumlah penjualan.

Jumlah data sebelum dan sesudah proses penghapusan duplikat dibandingkan untuk mengetahui berapa banyak baris yang berhasil dihapus.

Setelah itu, dilakukan pemeriksaan *missing values* pada kolom-kolom yang berkaitan langsung dengan pertanyaan analisis, yaitu:

- `order_id` untuk mengidentifikasi transaksi.
- `product_categories` untuk mengetahui kategori produk.
- `total_pembayaran` untuk menghitung pendapatan.
- `order_date` untuk menganalisis tren penjualan berdasarkan tahun.
- `kota_kabupaten` untuk melakukan analisis khusus pada wilayah Kabupaten Bogor.

Pemeriksaan ini dilakukan sebelum menghapus data kosong agar dapat ditentukan penanganan yang sesuai berdasarkan kebutuhan masing-masing analisis.

In [25]:
# Menyimpan jumlah data awal
jumlah_awal = len(df)

# Menghapus baris duplikat
df = df.drop_duplicates()

print("Duplikat dihapus:", jumlah_awal - len(df))

# Memeriksa missing values pada kolom utama analisis
kolom_penting = [
    "order_id",
    "product_categories",
    "total_pembayaran",
    "order_date",
    "kota_kabupaten"
]

print("\nMissing values kolom penting:")
display(df[kolom_penting].isnull().sum())

Duplikat dihapus: 0

Missing values kolom penting:


order_id                0
product_categories      0
total_pembayaran        0
order_date            188
kota_kabupaten          0
dtype: int64

# 5. Mengubah dan Memastikan Tipe Data

Setelah pemeriksaan *missing values*, tahap selanjutnya adalah memastikan setiap kolom memiliki tipe data yang sesuai dengan kebutuhan analisis.

Kolom `order_date` dan `waktu_pesanan_dibuat` diubah menjadi tipe data **datetime**. Perubahan ini diperlukan agar informasi tanggal dapat digunakan untuk mengambil tahun dan menganalisis tren penjualan dari waktu ke waktu.

Selain itu, kolom `total_qty`, `total_pembayaran`, `total_diskon`, dan `total_returned_qty` dipastikan memiliki tipe data numerik. Kolom-kolom tersebut akan digunakan dalam berbagai perhitungan pada tahap analisis.

Parameter `errors="coerce"` digunakan agar nilai yang tidak dapat dikonversi menjadi tanggal atau angka diubah menjadi nilai kosong (`NaT` untuk tanggal dan `NaN` untuk numerik). Nilai tersebut kemudian dapat diperiksa kembali sebelum menentukan tindakan selanjutnya.

Setelah proses konversi selesai, `df.info()` digunakan untuk memastikan bahwa tipe data pada kolom-kolom tersebut sudah sesuai.ya.

In [26]:
# Mengubah tanggal menjadi datetime
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df["waktu_pesanan_dibuat"] = pd.to_datetime(
    df["waktu_pesanan_dibuat"],
    errors="coerce"
)

# Memastikan kolom numerik bertipe angka
kolom_numerik = [
    "total_qty",
    "total_pembayaran",
    "total_diskon",
    "total_returned_qty"
]

for kolom in kolom_numerik:
    df[kolom] = pd.to_numeric(
        df[kolom],
        errors="coerce"
    )

# Melihat hasil
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18868 entries, 0 to 18867
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   order_id                            18868 non-null  str           
 1   total_qty                           18585 non-null  float64       
 2   total_weight_gr                     18868 non-null  int64         
 3   total_returned_qty                  18868 non-null  int64         
 4   total_diskon                        18868 non-null  int64         
 5   product_categories                  18868 non-null  str           
 6   num_product_categories              18868 non-null  int64         
 7   status_pesanan                      18868 non-null  str           
 8   alasan_pembatalan                   18868 non-null  str           
 9   opsi_pengiriman                     18868 non-null  str           
 10  metode_pembayaran                

# 6. Menentukan Transaksi Penjualan yang Valid

Setelah proses pemeriksaan dan penyesuaian data, dilakukan pemeriksaan terhadap variasi pada kolom `status_pesanan`. Langkah ini bertujuan untuk mengetahui kondisi transaksi yang terdapat dalam dataset, seperti pesanan yang selesai, dibatalkan, atau memiliki status lainnya.

Untuk analisis pendapatan dan penjualan, digunakan transaksi dengan status **`Selesai`** karena transaksi tersebut menunjukkan pesanan yang telah berhasil diselesaikan.

Transaksi dengan status selain `Selesai` tidak digunakan dalam dataset analisis penjualan agar pesanan yang dibatalkan tidak ikut dihitung sebagai pendapatan atau penjualan yang berhasil.

Data transaksi yang berstatus `Selesai` kemudian disimpan ke dalam DataFrame baru bernama `df_selesai`, sehingga dataset awal tetap tersedia untuk pemeriksaan atau analisis lainnya.

In [27]:
# Memeriksa variasi status setelah cleaning
print(df["status_pesanan"].value_counts())

# Mengambil transaksi selesai
df_selesai = df[
    df["status_pesanan"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("selesai")
].copy()

print("Jumlah pesanan selesai:", len(df_selesai))

status_pesanan
Selesai                                                                                   16045
Batal                                                                                      2573
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun P

# 7. Menyiapkan Data untuk Analisis

Setelah menentukan transaksi yang valid, data kemudian dipisahkan sesuai dengan kebutuhan masing-masing pertanyaan analisis.

### Analisis Tren Penjualan Tahunan

Untuk menganalisis tren penjualan setiap tahun, data yang memiliki nilai kosong pada `order_date` dan `total_pembayaran` tidak digunakan karena kedua kolom tersebut diperlukan untuk menghitung nilai penjualan berdasarkan waktu.

Kolom `order_date` kemudian digunakan untuk mengambil informasi tahun transaksi. Hasil tahun tersebut disimpan dalam kolom baru bernama `tahun`.

Dataset hasil penyaringan disimpan sebagai `df_tren` dan akan digunakan untuk mengetahui perubahan total penjualan dari tahun ke tahun.

### Analisis Kabupaten Bogor

Untuk menjawab pertanyaan mengenai kategori yang paling banyak dibeli di Kabupaten Bogor, dilakukan penyaringan berdasarkan kolom `kota_kabupaten`.

Data wilayah terlebih dahulu dibersihkan dari spasi yang tidak diperlukan dan diseragamkan menjadi huruf kapital. Selanjutnya, hanya transaksi dengan wilayah **`KAB. BOGOR`** yang dipilih.

Hasil penyaringan disimpan dalam DataFrame `df_bogor` dan akan digunakan untuk menganalisis kategori produk yang paling banyak dibeli di Kabupaten Bogor.

Jumlah data pada masing-masing DataFrame ditampilkan untuk memastikan proses penyaringan telah berjalan sesuai dengan kebutuhan analisis.

In [28]:
# Data untuk analisis tren tahunan
df_tren = df_selesai.dropna(
    subset=["order_date", "total_pembayaran"]
).copy()

df_tren["tahun"] = df_tren["order_date"].dt.year

# Data untuk analisis wilayah Kabupaten Bogor
df_bogor = df_selesai[
    df_selesai["kota_kabupaten"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("KAB. BOGOR")
].copy()

print("Data tren tahunan:", len(df_tren))
print("Data Kabupaten Bogor:", len(df_bogor))

Data tren tahunan: 15541
Data Kabupaten Bogor: 955


# 8. Menyimpan Dataset Hasil Cleaning

Setelah seluruh proses *data cleaning* selesai, dataset yang telah dibersihkan disimpan ke dalam file CSV baru dengan nama `indonesia_ecommerce_sales_clean.csv`.

Dataset utama `df` disimpan agar hasil pembersihan dapat digunakan kembali pada tahap **Exploratory Data Analysis (EDA)** dan analisis selanjutnya tanpa perlu mengulangi proses *cleaning* dari awal.

Parameter `index=False` digunakan agar indeks DataFrame tidak ikut disimpan sebagai kolom tambahan pada file CSV.

Ukuran dataset setelah proses *cleaning* juga ditampilkan untuk memastikan jumlah baris dan kolom pada data akhir.

In [29]:
# Simpan dataset bersih
df.to_csv(
    "indonesia_ecommerce_sales_clean.csv",
    index=False
)

print("Dataset bersih berhasil disimpan.")
print("Ukuran akhir:", df.shape)

Dataset bersih berhasil disimpan.
Ukuran akhir: (18868, 19)
